In [1]:
import requests
import json

url = "https://gamma-api.polymarket.com/markets"
params = {"closed": "true", "limit": 5}
resp = requests.get(url, params=params, timeout=30)
data = resp.json()

# Dump the full structure of the first one
print(json.dumps(data[0], indent=2, default=str))

{
  "id": "12",
  "question": "Will Joe Biden get Coronavirus before the election?",
  "conditionId": "0xe3b423dfad8c22ff75c9899c4e8176f628cf4ad4caa00481764d320e7415f7a9",
  "slug": "will-joe-biden-get-coronavirus-before-the-election",
  "twitterCardImage": "https://polymarket-static.s3.us-east-2.amazonaws.com/polymarket-twitter-card.png",
  "endDate": "2020-11-04T00:00:00Z",
  "category": "US-current-affairs",
  "liquidity": "0",
  "image": "https://polymarket-upload.s3.us-east-2.amazonaws.com/will-joe-biden-get-coronavirus-before-the-election-5a3d4c3b-0a73-419e-a686-be034d2c73ac.png",
  "icon": "https://polymarket-upload.s3.us-east-2.amazonaws.com/will-joe-biden-get-coronavirus-before-the-election-5a3d4c3b-0a73-419e-a686-be034d2c73ac.png",
  "description": "This is a market on if presidential candidate Joe Biden will test positive for COVID-19 before November 3rd, 2020. This will be defined as Joe publicly announcing - via any medium, including social media - that he has tested posit

In [2]:
import requests
from collections import Counter

url = "https://gamma-api.polymarket.com/markets"
resp = requests.get(url, params={"closed": "true", "limit": 200}, timeout=30)
data = resp.json()

patterns = Counter()
for m in data:
    patterns[m.get("outcomePrices")] += 1

for pattern, count in patterns.most_common(20):
    print(f"{count:4d}  {pattern}")

  45  ["0", "0"]
   1  ["0.000001011082052522541417308141468657552", "0.9999989889179474774585826918585313"]
   1  ["0.000001024519509568169644816863666886675", "0.9999989754804904318303551831363331"]
   1  ["0.00000004364303498046286702037228176483457", "0.9999999563569650195371329796277182"]
   1  ["0.5843311669077693146635445446077281", "0.4156688330922306853364554553922719"]
   1  ["0.4988929462743076960451117056339252", "0.5011070537256923039548882943660748"]
   1  ["0.9999688709703679916078209460980032", "0.00003112902963200839217905390199679565"]
   1  ["0.0000002365760136311761420505896616707271", "0.9999997634239863688238579494103383"]
   1  ["0.9999928071628365951114292824952643", "0.000007192837163404888570717504735668924"]
   1  ["0.000001969126198486787442291877571524744", "0.9999980308738015132125577081224285"]
   1  ["0.9999991353509498666949062749530927", "0.0000008646490501333050937250469073407429"]
   1  ["0.9999998888806886895302218251805338", "0.00000011111931131046

In [3]:
import requests, json

url = "https://gamma-api.polymarket.com/markets"
resp = requests.get(url, params={"closed": "true", "limit": 200}, timeout=30)
data = resp.json()

for m in data[:30]:
    prices = m.get("outcomePrices")
    print(
        "uma:", repr(m.get("umaResolutionStatuses")),
        "| closed:", m.get("closed"),
        "| acceptingOrders:", m.get("acceptingOrders"),
        "| prices:", prices,
    )

uma: '[]' | closed: True | acceptingOrders: None | prices: ["0", "0"]
uma: '[]' | closed: True | acceptingOrders: None | prices: ["0", "0"]
uma: '[]' | closed: True | acceptingOrders: None | prices: ["0", "0"]
uma: '[]' | closed: True | acceptingOrders: None | prices: ["0.000001011082052522541417308141468657552", "0.9999989889179474774585826918585313"]
uma: '[]' | closed: True | acceptingOrders: None | prices: ["0.000001024519509568169644816863666886675", "0.9999989754804904318303551831363331"]
uma: '[]' | closed: True | acceptingOrders: None | prices: ["0", "0"]
uma: '[]' | closed: True | acceptingOrders: None | prices: ["0.00000004364303498046286702037228176483457", "0.9999999563569650195371329796277182"]
uma: '[]' | closed: True | acceptingOrders: None | prices: ["0", "0"]
uma: '[]' | closed: True | acceptingOrders: None | prices: ["0.5843311669077693146635445446077281", "0.4156688330922306853364554553922719"]
uma: '[]' | closed: True | acceptingOrders: None | prices: ["0.4988929462

In [6]:
import pandas as pd
from db.connection import get_conn

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

with get_conn() as conn:
    # 1. A few full market rows — check every column landed correctly
    markets = pd.read_sql("""
        SELECT market_id, title, active, closed, resolved, resolved_outcome,
               volume, volume_24h, liquidity, best_bid, best_ask, mid_price, spread,
               event_time, updated_at
        FROM markets
        ORDER BY updated_at DESC
        LIMIT 5
    """, conn)

    # 2. The resolution cross-tab — sanity-check the new columns
    crosstab = pd.read_sql("""
        SELECT closed, resolved, resolved_outcome, COUNT(*) AS n
        FROM markets
        GROUP BY closed, resolved, resolved_outcome
        ORDER BY closed, resolved, resolved_outcome
    """, conn)

    # 3. A few recent snapshots — confirm the hot path still writes cleanly
    snaps = pd.read_sql("""
        SELECT market_id, best_bid, best_ask, mid_price, spread,
               volume_24h, liquidity, active, snapshot_at
        FROM market_snapshots
        ORDER BY snapshot_at DESC
        LIMIT 5
    """, conn)

print("=== MARKETS (5 most recently updated) ===")
print(markets.to_string(index=False))
print("\n=== RESOLUTION CROSS-TAB ===")
print(crosstab.to_string(index=False))
print("\n=== RECENT SNAPSHOTS ===")
print(snaps.to_string(index=False))

=== MARKETS (5 most recently updated) ===
market_id                                                                      title  active  closed  resolved resolved_outcome       volume  volume_24h  liquidity  best_bid  best_ask  mid_price  spread                event_time                       updated_at
  2000268 Will Justin Bieber perform at the 2026 FIFA World Cup Final halftime show?    True   False     False             None 10158.628395  554.526106  216.27540     0.440     0.600     0.5200   0.160 2026-07-19 00:00:00+00:00 2026-06-19 18:25:11.397235+00:00
  2000266         Will Jay-Z perform at the 2026 FIFA World Cup Final halftime show?    True   False     False             None   278.731300         NaN  632.85277     0.013     0.030     0.0215   0.017 2026-07-19 00:00:00+00:00 2026-06-19 18:25:11.397230+00:00
  2000264  Will Travis Scott perform at the 2026 FIFA World Cup Final halftime show?    True   False     False             None  3495.845095   20.330000 1542.93809     0.03

/tmp/ipykernel_152788/3141257059.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  markets = pd.read_sql("""
/tmp/ipykernel_152788/3141257059.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  crosstab = pd.read_sql("""
/tmp/ipykernel_152788/3141257059.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  snaps = pd.read_sql("""


In [5]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
print(sys.path[0])

/home/benjamin-rodriguez/Desktop/PredictionMarkets
